# 21장 · 문서 청킹과 원문 오프셋

20장에서 읽은 도서관 안내를 나누고, ‘24시간 이용할 수’가 어느 청크에 남는지 원문 위치로 확인합니다. 분관 시간표에서는 머리글과 운영 시간 행을 함께 검색할 수 있는 경계를 찾습니다.

실행을 확인한 커널은 Python 3.12.15입니다. 예제에 사용한 외부 패키지는 langchain-core 1.6.12, langchain-text-splitters 1.1.3입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 과제 셀에 직접 코드를 작성한 뒤 바로 아래에서 풀이를 펼쳐 보세요.

## 쪽 단위 입력과 고정 길이 청킹

청크는 검색기에 넣을 문서 조각입니다. 먼저 어느 원문을 잘랐는지 정해 두세요. 20장의 세 쪽을 합쳐 버리면 청크 안의 문장이 원래 PDF 몇 쪽에 있었는지 되찾기 어렵습니다. 길이는 파이썬 len()으로 잰 문자 수입니다.

노트북에서 패키지를 찾을 수 없다는 오류가 나면 새 셀에 %pip install langchain-core==1.6.12 langchain-text-splitters==1.1.3을 실행한 뒤 커널을 다시 시작하세요. 실습에 외부 모델 API 키는 필요하지 않습니다.

### 셀 01 · 20장 이용 안내를 쪽별 Document로 열기

guide-records.json은 20장 셀 04에서 얻은 세 레코드입니다. 노트북만 열어도 같은 값으로 시작합니다. 3쪽의 본문과 출처를 먼저 읽어 보세요.

In [1]:
from pathlib import Path
import json
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

records_path = Path("guide-records.json")
records = [
    {"id": "library-1", "text": "도서 대출 기간은 14일입니다.", "source": "가상 도서관 이용 안내", "page": 1},
    {"id": "library-2", "text": "도서관은 월요일에 휴관합니다.", "source": "가상 도서관 이용 안내", "page": 2},
    {"id": "library-3", "text": "무인 반납함은 24시간 이용할 수 있습니다.", "source": "가상 도서관 이용 안내", "page": 3},
]
if records_path.is_file():
    records = json.loads(records_path.read_text(encoding="utf-8"))
pages = [
    Document(page_content=r["text"], metadata={
        "document_id": r["id"], "source": r["source"], "page": r["page"]
    })
    for r in records
]
page = pages[2]
print(page.page_content)
print(page.metadata)

무인 반납함은 24시간 이용할 수 있습니다.
{'document_id': 'library-3', 'source': '가상 도서관 이용 안내', 'page': 3}


library-3은 20장에서 library-guide.pdf의 3쪽에 붙인 ID입니다. source에는 읽을 사람이 볼 자료명이, page에는 원래 쪽 번호가 남습니다.

원문 3쪽에는 반납함의 이름과 이용 시간이 함께 있습니다. 이 문장을 둘로 나눠 ‘24시간’만 검색되면 반납함의 이용 시간이라는 문맥이 빠집니다. 청크 경계를 바꿀 때는 숫자뿐 아니라 그 숫자가 가리키는 대상까지 한 조각에서 읽히는지 보세요.


### 셀 02 · 고정 길이에서 잘리는 대출 기간

1쪽을 공백과 온점까지 세어 16자에서 자릅니다. 둘째 조각을 첫 조각의 끝에서 네 글자 앞으로 당겨 보세요.

In [2]:
text = pages[0].page_content
first = text[:16]
second = text[12:]
print(first)
print(second)
print(text[16:])

도서 대출 기간은 14일입니다
일입니다.
.


첫 조각에는 마침표가 없습니다. 겹침을 둔 둘째 조각은 ‘일입니다.’이고 겹침 없이 잘랐다면 ‘.’만 남습니다. 12와 16은 이 고정 폭 실험의 좌표입니다. 다음 분할기는 공백도 경계 후보로 씁니다.

## 재귀 분할과 실제 겹침

RecursiveCharacterTextSplitter는 기본 설정에서 빈 줄, 줄바꿈, 공백, 개별 글자 순으로 경계를 찾습니다. 큰 단위가 한도 안에 들어가면 유지하고, 너무 긴 부분만 다음 구분자로 다시 나눕니다. 조각의 위치까지 보려면 생성할 때 add_start_index=True를 지정하세요.

### 셀 03 · chunk_size=12에서 끊기는 구절

3쪽에는 “24시간 이용할 수”가 연달아 나옵니다. 같은 구절을 적어도 한 청크 안에서 그대로 찾을 수 있는지 확인하세요. split_documents()에 쪽별 Document를 넣으면 조각마다 원본의 메타데이터가 따라옵니다.

In [3]:
def split_pages(documents, size, overlap):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=size, chunk_overlap=overlap, add_start_index=True
    )
    return splitter.split_documents(documents)

baseline = split_pages([page], size=12, overlap=0)
for piece in baseline:
    print(piece.metadata["start_index"], repr(piece.page_content))
phrase = "24시간 이용할 수"
print(any(phrase in piece.page_content for piece in baseline))

0 '무인 반납함은 24시간'
13 '이용할 수 있습니다.'
False


첫 조각은 ‘24시간’에서 끝나고 둘째는 ‘이용할’에서 시작합니다. False는 이 구절 전체를 담은 조각이 없다는 뜻입니다. 이 검사로 실제 검색 순위나 답변 정확도까지 알 수는 없습니다.

### 과제 04 · chunk_overlap을 4자로 바꿔 실제 겹침 검산하기

RecursiveCharacterTextSplitter를 직접 만들고 3쪽에 split_documents()를 호출하세요. 크기는 12로 두고 overlap만 4로 바꾸세요. 첫 청크의 끝과 다음 청크의 시작을 같은 쪽의 문자 위치로 비교해 실제로 공유하는 글자 수를 구하세요. 끝 위치는 해당 청크에 포함되지 않습니다.

In [4]:
# size 12, overlap 4로 분할기를 만들고 시작 위치를 기록하도록 설정하세요.
# 3쪽을 분할해 두 청크가 겹치는 글자 수를 출력하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>splitter = RecursiveCharacterTextSplitter(
    chunk_size=12, chunk_overlap=4, add_start_index=True
)
trial = splitter.split_documents([page])
left, right = trial
end = left.metadata[&quot;start_index&quot;] + len(left.page_content)
start = right.metadata[&quot;start_index&quot;]
print(end, start, max(0, end - start))</code></pre>
<p>실행 결과</p><pre><code>12 13 0</code></pre>
<p>끝은 12, 다음 시작은 13이므로 실제 겹침은 0자입니다. 사이의 한 칸은 공백입니다. 분할기가 조합하는 단위에는 앞 공백도 포함될 수 있고, 청크 양 끝의 공백은 기본 설정에서 제거됩니다. 설정한 4자가 두 문자열에 그대로 남는다고 가정하면 틀립니다.</p>
</details>

### 셀 05 · 겹침을 6자로 바꿔 같은 구절 다시 찾기

한 번 더 overlap만 바꿉니다. 근거 구절이 통째로 포함된 청크를 골라 그 위치와 본문을 읽으세요.

In [5]:
repaired = split_pages([page], size=12, overlap=6)
hit = next(p for p in repaired if phrase in p.page_content)
start = hit.metadata["start_index"]
print(start, start + len(hit.page_content), hit.page_content)

8 18 24시간 이용할 수


8에서 시작해 18 직전까지 읽으면 ‘24시간 이용할 수’입니다. 첫 조각의 ‘24시간’을 둘째 조각에서도 읽을 수 있게 됐습니다. 이 설정에서는 청크 수도 둘에서 셋으로 늘었습니다. 첫 조각과 둘째 조각이 공유하는 실제 원문은 8 이상 12 미만의 네 글자입니다.

[웹 교재에서 같은 세 쪽의 청크 경계 실험 열기](https://langchain-lecture.vercel.app/chapter21/index.html#practice-space)

## 청크에서 원문 위치 복원

start_index는 한 쪽의 page_content 안에서 센 문자 위치입니다. 같은 숫자 10이 두 쪽에 나와도 서로 다른 위치입니다. 청크를 확인할 때는 먼저 document_id로 쪽을 찾고, 그 본문의 [start:end]를 잘라 청크와 대조합니다.

### 과제 06 · 출처와 본문을 대조하는 함수

앞에서 얻은 hit를 입력으로 받습니다. 문서 ID로 원래 쪽을 찾고, 문자 위치로 되짚어 슬라이스가 청크와 같은지 검사하세요. 위치가 틀렸으면 AssertionError가 나도록 작성하세요.

In [6]:
parents = {doc.metadata["document_id"]: doc for doc in pages}

def source_span(piece, parents):
    # 원문을 찾아 본문과 위치, 쪽을 검산하세요.
    pass

# print(source_span(hit, parents))

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>parents = {doc.metadata[&quot;document_id&quot;]: doc for doc in pages}

def source_span(piece, parents):
    meta = piece.metadata
    parent = parents[meta[&quot;document_id&quot;]]
    start = meta[&quot;start_index&quot;]
    end = start + len(piece.page_content)
    assert 0 &lt;= start &lt; end &lt;= len(parent.page_content)
    assert parent.page_content[start:end] == piece.page_content
    assert (meta[&quot;source&quot;], meta[&quot;page&quot;]) == (parent.metadata[&quot;source&quot;], parent.metadata[&quot;page&quot;])
    return meta[&quot;document_id&quot;], meta[&quot;page&quot;], start, end

print(source_span(hit, parents))</code></pre>
<p>실행 결과</p><pre><code>(&#x27;library-3&#x27;, 3, 8, 18)</code></pre>
<p>library-3의 3쪽에서 8 이상 18 미만을 읽습니다. hit를 복사해 start_index를 9로 바꾼 뒤 함수가 실패하는지 확인해 보세요. 청크 본문이 우연히 여러 번 반복되면 문자열 일치만으로 잘못 붙은 쪽을 가려낼 수 없어 출처와 쪽도 확인합니다.</p>
</details>

## 두 쪽짜리 안내문의 문단과 운영 시간표

새솔도서관 분관 안내는 수업을 위해 만든 텍스트입니다. 두 쪽에 같은 제목과 ‘운영 시간’ 표제가 나오지만 각 표의 대상과 시간은 다릅니다. 목표는 “토요일 안내 데스크는 몇 시에 닫나요?”에 답할 수 있는 표 조각을 찾고 근거를 원문에서 대조하는 것입니다.

### 셀 07 · 문단과 표의 줄바꿈을 보존해 열기

branch-guide.txt는 쪽 사이에 폼 피드(\f) 한 글자를 둔 가상 문서입니다. 내려받은 파일이 있으면 우선해서 읽습니다. 노트북에 들어 있는 사본도 같은 원문입니다. 표의 열과 두 쪽의 순서를 눈으로 확인하세요.

In [7]:
sample_pages = (
    """새솔도서관 분관 이용 안내
2027년 3월 연습용 개정판

자료 예약
예약 도서가 도착하면 문자로 안내합니다. 안내 문자를 받은 날부터 사흘 동안 1층 대출대에서 책을 찾아가세요. 기한이 지나면 다음 예약자에게 순서가 넘어갑니다.

운영 시간
장소 | 평일 | 토요일
대출대 | 09:00~20:00 | 09:00~16:00
자료실 | 09:00~21:00 | 09:00~17:00

휴관 안내
매주 월요일에는 자료실과 대출대를 닫습니다. 휴관 중에도 무인 반납함에 책을 넣을 수 있습니다.""",
    """새솔도서관 분관 이용 안내
2027년 3월 연습용 개정판

무인 반납함
중앙 출입구 오른쪽의 반납함은 매일 24시간 이용할 수 있습니다. 투입구가 가득 찼을 때에는 책을 바닥에 놓지 말고 다음 개관일에 대출대로 가져오세요.

분실물 접수
개인 소지품을 습득하면 2층 안내 데스크에 맡겨 주세요. 접수한 물건은 보름 동안 보관합니다. 찾으러 올 때에는 물건의 색상과 습득 장소를 직원에게 알려 주세요.

운영 시간
장소 | 평일 | 토요일
안내 데스크 | 10:00~18:00 | 10:00~15:00
무인 반납함 | 24시간 | 24시간"""
)
branch_path = Path("branch-guide.txt")
raw = branch_path.read_text(encoding="utf-8") if branch_path.is_file() else "\f".join(sample_pages)
branch = [Document(page_content=body, metadata={
    "document_id": f"branch-{number}", "source": "branch-guide.txt", "page": number
}) for number, body in enumerate(raw.split("\f"), start=1)]
print([len(p.page_content) for p in branch])
print(branch[1].page_content.split("운영 시간\n")[-1])

[274, 300]
장소 | 평일 | 토요일
안내 데스크 | 10:00~18:00 | 10:00~15:00
무인 반납함 | 24시간 | 24시간


쪽 구분자를 빼고 세면 1쪽은 274자, 2쪽은 300자입니다. 마지막 표의 열은 장소·평일·토요일 순서입니다. 답을 읽을 때에는 안내 데스크 행의 마지막 열을 봅니다. 분할 전에 모든 공백을 하나로 합치면 줄과 표의 경계를 잃습니다.

### 셀 08 · 55자로 분할한 시간표에서 빠진 머리글

분관 두 쪽을 각각 size=55, overlap=12로 자릅니다. “안내 데스크”가 분실물 설명에도 등장하므로 표의 행 구분 기호까지 넣어 찾습니다.

In [8]:
def desk_row(chunks):
    return next(c for c in chunks if "안내 데스크 |" in c.page_content)

short_chunks = split_pages(branch, size=55, overlap=12)
short_row = desk_row(short_chunks)
print(short_row.metadata)
print(short_row.page_content)

{'document_id': 'branch-2', 'source': 'branch-guide.txt', 'page': 2, 'start_index': 245}
안내 데스크 | 10:00~18:00 | 10:00~15:00


분관 2쪽의 위치 245에서 시작하는 청크에는 시간 두 개가 있지만 열 이름은 없습니다. 원문으로 돌아가 바로 위 머리글을 읽어야 10:00~15:00이 토요일 시간인지 확인할 수 있습니다.

### 과제 09 · 표 머리글과 시간 행이 함께 남는 크기 찾기

같은 두 쪽과 겹침 12를 유지하고 크기 후보 54, 70을 직접 시험하세요. 각 후보에서 안내 데스크 행을 골라 머리글 ‘장소 | 평일 | 토요일’과 토요일 범위 ‘10:00~15:00’이 한 청크에 있는지 검사하세요. 시험한 후보 중 두 조건을 만족하는 가장 작은 크기를 택하고, 과제 06의 source_span()으로 그 행을 원본에서 찾아 출력하세요.

In [9]:
# 두 후보에서 분할기를 실행하고 머리글과 행을 함께 보존한 크기를 고르세요.
# 선택한 청크의 원문 ID, 쪽과 구간을 대조하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>accepted = {}
for size in (54, 70):
    trial = split_pages(branch, size=size, overlap=12)
    candidate = desk_row(trial)
    intact = (&quot;장소 | 평일 | 토요일&quot; in candidate.page_content
              and &quot;10:00~15:00&quot; in candidate.page_content)
    print(size, intact)
    if intact:
        accepted[size] = candidate

selected_size = min(accepted)
row = accepted[selected_size]
roots = {p.metadata[&quot;document_id&quot;]: p for p in branch}
doc_id, page_number, start, end = source_span(row, roots)
print(doc_id, page_number, start, end)
print(roots[doc_id].page_content[start:end])</code></pre>
<p>실행 결과</p><pre><code>54 False
70 True
branch-2 2 225 279
운영 시간
장소 | 평일 | 토요일
안내 데스크 | 10:00~18:00 | 10:00~15:00</code></pre>
<p>54에서는 머리글이 떨어졌고, 70에서는 branch-2의 225 이상 279 미만에 머리글과 행이 함께 남습니다. 마지막 열에서 토요일에 닫는 시각 15:00을 읽을 수 있습니다. 70은 시험한 두 후보 중에서 고른 값입니다. 54와 70 사이의 크기를 추가로 시험해도 같은 기준으로 판단할 수 있습니다. 1쪽에는 같은 표제가 있지만 대출대와 자료실의 시간만 나옵니다.</p>
</details>

### 과제 10 · 긴 문단에서 실제로 겹친 글자 읽기

55자 실험에서 1쪽 예약 규칙을 담은 두 청크를 고르세요. 두 청크의 원문 구간이 겹치는 부분을 구해 문자열과 길이를 출력하세요. 2쪽 시간표에서 머리글과 행 사이에도 같은 길이만큼 겹치는지 비교하세요.

In [10]:
# "예약 도서가"가 있는 청크와 바로 다음 청크의 교집합을 구하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>i = next(i for i, c in enumerate(short_chunks) if &quot;예약 도서가&quot; in c.page_content)
left, right = short_chunks[i:i + 2]
end = left.metadata[&quot;start_index&quot;] + len(left.page_content)
begin = right.metadata[&quot;start_index&quot;]
shared = branch[0].page_content[begin:end]
print(repr(shared), len(shared))
j = short_chunks.index(short_row)
heading = short_chunks[j - 1]
heading_end = heading.metadata[&quot;start_index&quot;] + len(heading.page_content)
print(max(0, heading_end - short_row.metadata[&quot;start_index&quot;]))</code></pre>
<p>실행 결과</p><pre><code>&#x27;1층 대출대에서 책을&#x27; 11
0</code></pre>
<p>1쪽 예약 문단에는 ‘1층 대출대에서 책을’ 11글자가 두 번 들어 있고, 2쪽 시간표의 머리글과 행은 0글자가 겹칩니다. 같은 12자를 설정해도 실제 경계는 문단과 표에서 다르게 생깁니다. 운영 문서에서 머리글이 필수라면 크기를 바꾼 뒤에도 각 행에 질문의 전제가 남았는지 다시 읽어야 합니다.</p>
</details>

## 인계용 청크와 기준 문서

실험 결과를 넘길 때는 원문의 ID와 청크의 ID를 구별합니다. 같은 파일도 분할 설정을 바꾸면 순번과 본문이 달라집니다. 22장의 거리 비교는 library-1·2·3 세 ID와 쪽 전체를 사용합니다. 두 쪽짜리 분관 안내와 아래 실험용 청크는 별도로 보관합니다.

### 과제 11 · 동료가 원문을 찾을 수 있는 청크 저장하기

과제 09에서 선택한 분관 2쪽의 표 청크를 JSON에 저장하세요. 필요한 값은 청크 ID(원문 ID와 시작점 조합), 원문 ID, 본문, 출처, 쪽, 시작·끝 위치, 분할 설정입니다. 다시 읽은 JSON만 보고 동료가 같은 원문을 잘라 낼 수 있어야 합니다.

In [11]:
# 과제 09에서 선택한 시간 행의 위치를 검산하세요.
# chapter21-output/desk-hours.json에 저장하고 다시 읽어 본문을 확인하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>piece = row
doc_id, number, start, end = source_span(piece, roots)
record = {&quot;id&quot;: f&quot;{doc_id}:{start}&quot;, &quot;document_id&quot;: doc_id, &quot;text&quot;: piece.page_content,
          &quot;source&quot;: piece.metadata[&quot;source&quot;], &quot;page&quot;: number, &quot;start&quot;: start, &quot;end&quot;: end,
          &quot;chunk_size&quot;: selected_size, &quot;chunk_overlap&quot;: 12}
path = Path(&quot;chapter21-output/desk-hours.json&quot;)
path.parent.mkdir(exist_ok=True)
path.write_text(json.dumps(record, ensure_ascii=False, indent=2), encoding=&quot;utf-8&quot;)
saved = json.loads(path.read_text(encoding=&quot;utf-8&quot;))
original = roots[saved[&quot;document_id&quot;]].page_content
assert original[saved[&quot;start&quot;]:saved[&quot;end&quot;]] == saved[&quot;text&quot;]
print(saved[&quot;id&quot;], saved[&quot;source&quot;], saved[&quot;page&quot;])
print(saved[&quot;text&quot;])</code></pre>
<p>실행 결과</p><pre><code>branch-2:225 branch-guide.txt 2
운영 시간
장소 | 평일 | 토요일
안내 데스크 | 10:00~18:00 | 10:00~15:00</code></pre>
<p>branch-2:225는 분관 2쪽의 시작 위치를 기록한 ID입니다. 크기를 바꾸면 시작점이 같더라도 끝 위치가 달라질 수 있습니다. 다른 분할본을 대조할 때는 ID와 함께 end, 분할 설정, 잘라낸 본문까지 확인하세요.</p>
</details>

### 셀 12 · 22장의 세 기준 ID와 선택 인계 파일 읽기

22장은 세 문장에 벡터를 하나씩 붙입니다. 기준 자료의 ID와 쪽을 출력하세요. 20장 과제의 desk-handoff.json이 있으면 노트북 옆 chapter20-output 폴더에 두세요. 선택 자료는 따로 읽어 TXT의 page가 어떤 값인지 확인합니다.

In [12]:
print([(r["id"], r["page"]) for r in records])
desk_path = Path("chapter20-output/desk-handoff.json")
if desk_path.is_file():
    transferred = json.loads(desk_path.read_text(encoding="utf-8"))
    desk_doc = transferred["documents"][0]
    print(desk_doc["id"], desk_doc["source"], desk_doc["page"], desk_doc["text"])

[('library-1', 1), ('library-2', 2), ('library-3', 3)]


기준 출력은 1·2·3쪽과 library-1·2·3입니다. 선택 파일도 열었다면 desk-1, desk-change.txt와 쪽 번호 None을 한 줄 더 읽게 됩니다. 이 문서를 분할할 때도 파일의 첫 글자부터 위치를 세면 됩니다.

## 돌아보기

처음 보는 문서 한 건을 짝과 바꿔 읽으세요. 긴 문단이나 표에서 필요한 근거 구절을 먼저 정하고, 같은 쪽을 두 설정으로 나누어 그 구절과 전제가 한 청크에 남았는지 확인하세요. 상대에게 원문과 청크 JSON을 건네고, 기록한 쪽과 위치로 되짚은 문장이 같은지 검산해 보세요.